In [ ]:
from pathlib import Path
import time
import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import matplotlib.pyplot as plt
from scipy import stats
from scipy.sparse import csr_matrix
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.metrics import silhouette_score

# ---------------- parameters (change here, then Run All) ----------------
REAL_PCT = 50            # real listen = at least this % played
N_TRACKS = 100           # distinct tracks per user and stream
SESSION_GAP_S = 30 * 60
K_GENRE = 32             # number of pseudo-genres (k-means clusters)
CAP = 15                 # session baskets: max distinct tracks per basket
MAX_BASKETS_PER_USER = 100
SEED = 42

rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
COL = {"organic": "#2a9d8f", "recommended": "#e76f51"}

ROOT = Path.cwd()
while not (ROOT / "data" / "raw" / "yambda").exists():
    if ROOT.parent == ROOT:
        raise FileNotFoundError("Could not find data/raw/yambda above " + str(Path.cwd()))
    ROOT = ROOT.parent
DATA, PROC = ROOT / "data" / "raw" / "yambda", ROOT / "data" / "processed"
verdict = []


def add_verdict(method, metric, organic, recommended, higher_means):
    rec_higher = recommended > organic
    broader = rec_higher if higher_means == "broader" else not rec_higher
    verdict.append({"analysis": method, "metric": metric, "organic": organic, "recommended": recommended,
                    "relative difference": (recommended - organic) / abs(organic) if organic else np.nan,
                    "recommended is": "broader" if broader else "narrower"})


def rank_in_group(g):
    order = np.argsort(g, kind="stable"); gs = g[order]
    start = np.r_[0, np.flatnonzero(gs[1:] != gs[:-1]) + 1]
    sizes = np.diff(np.r_[start, len(gs)])
    rank = np.empty(len(g), dtype=np.int64)
    rank[order] = np.arange(len(gs)) - np.repeat(start, sizes)
    return rank


def paired(org, rec):
    """Paired comparison over users: medians, bootstrap CI of the median difference, share, Wilcoxon p."""
    d = rec - org
    boot = np.median(d[rng.integers(0, len(d), (1000, len(d)))], axis=1)
    return dict(users=len(d), organic=np.median(org), recommended=np.median(rec), diff=np.median(d),
                lo=np.quantile(boot, 0.025), hi=np.quantile(boot, 0.975),
                share_rec_higher=(d > 0).mean(), p=stats.wilcoxon(d).pvalue)

print("project root:", ROOT)

In [ ]:
def same_group_pairs(label, n_pairs=200_000):
    lab = label[emb_ids]
    order = np.argsort(lab, kind="stable"); ls = lab[order]
    same = (ls[1:] == ls[:-1]) & (ls[1:] >= 0)
    i1, i2 = order[:-1][same], order[1:][same]
    pick = rng.choice(len(i1), min(n_pairs, len(i1)), replace=False)
    return np.einsum("ij,ij->i", EN[i1[pick]], EN[i2[pick]])

cos_artist, cos_album = same_group_pairs(item_art), same_group_pairs(item_alb)
r1, r2 = rng.integers(0, len(EN), 200_000), rng.integers(0, len(EN), 200_000)
cos_rand = np.einsum("ij,ij->i", EN[r1], EN[r2])

def auc(pos, neg):
    r = stats.rankdata(np.r_[pos, neg]); return (r[:len(pos)].sum() - len(pos) * (len(pos) + 1) / 2) / (len(pos) * len(neg))

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
bins = np.linspace(-0.4, 1, 80)
for name, v, c in [("random pair", cos_rand, "#6c757d"), ("same artist", cos_artist, "#264653"), ("same album", cos_album, "#e9c46a")]:
    ax[0].hist(v, bins=bins, histtype="step", density=True, lw=1.6, label=f"{name} (mean {v.mean():.2f})", color=c)
ax[0].set(title="Cosine similarity of track pairs", xlabel="cosine similarity", ylabel="density"); ax[0].legend()
sub = rng.choice(len(EN), 60_000, replace=False)
xy = PCA(n_components=2, random_state=SEED).fit_transform(EN[sub])
hb = ax[1].hexbin(xy[:, 0], xy[:, 1], C=np.log10(np.maximum(item_users[emb_ids[sub]], 1)), reduce_C_function=np.mean,
                  gridsize=45, cmap="viridis", mincnt=5)
plt.colorbar(hb, ax=ax[1], label="mean log10(users per track)")
ax[1].set(title="Map of the sound space (PCA of 60k tracks)", xlabel="PC1", ylabel="PC2")
plt.tight_layout(); plt.show()
print(f"AUC same-artist vs random: {auc(cos_artist, cos_rand):.3f}   same-album vs random: {auc(cos_album, cos_rand):.3f}   (0.5 = no signal)")

In [ ]:
Co, Cr = centroid[:, 1], centroid[:, 0]
iu = np.triu_indices(len(users), 1)
sim = {"organic": Co @ Co.T, "recommended": Cr @ Cr.T}
vals = {s: sim[s][iu] for s in sim}
rows = []
for s in sim:
    v = vals[s]; M_ = sim[s].copy(); np.fill_diagonal(M_, -1)
    rows.append({"stream": s, "mean cosine": v.mean(), "p50": np.median(v), "p99": np.quantile(v, 0.99),
                 "share >= 0.9": (v >= 0.9).mean(), "share >= 0.95": (v >= 0.95).mean(), "median nearest-neighbour cosine": np.median(M_.max(axis=1))})
T6 = pd.DataFrame(rows).set_index("stream")
print(T6.to_string(float_format=lambda x: f"{x:.4f}"))
idx = rng.choice(len(vals["organic"]), 400_000, replace=False)
dd = vals["recommended"][idx] - vals["organic"][idx]
print(f"\nsame pair of users: recommended centroids more similar in {(dd > 0).mean():.1%} of pairs (mean difference {dd.mean():+.4f}, p = {stats.wilcoxon(dd).pvalue:.1e})")
nn = {s: (lambda M_: (np.fill_diagonal(M_, -1), M_.max(axis=1))[1])(sim[s].copy()) for s in sim}
res6 = paired(nn["organic"], nn["recommended"])
print(f"nearest-neighbour cosine per user: organic {res6['organic']:.4f}, recommended {res6['recommended']:.4f}, "
      f"{res6['share_rec_higher']:.1%} of users have a closer twin in the recommended stream (p = {res6['p']:.1e})")
# how far apart are a user's own two centroids, compared with two random users?
own = np.einsum("ij,ij->i", Co, Cr)
rand_pair = np.einsum("ij,ij->i", Co, Cr[rng.permutation(len(Cr))])
print(f"cosine between a user's own organic and recommended centroid: {np.median(own):.3f}; organic user vs random other user's recommended: {np.median(rand_pair):.3f}")

fig, ax = plt.subplots(1, 2, figsize=(12, 3.6))
for s in sim:
    ax[0].hist(vals[s], bins=np.linspace(0.3, 1, 80), histtype="step", color=COL[s], lw=1.6, label=f"{s} (mean {vals[s].mean():.3f})", density=True)
ax[0].set(yscale="log", title="Cosine similarity of user taste centroids", xlabel="cosine", ylabel="density"); ax[0].legend()
ax[1].hist(own, bins=40, alpha=0.8, color="#264653", label="own organic vs own recommended")
ax[1].hist(rand_pair, bins=40, alpha=0.6, color="#adb5bd", label="organic vs another user's recommended")
ax[1].set(title="Is a user's recommended taste close to their organic taste?", xlabel="cosine"); ax[1].legend(fontsize=8)
plt.tight_layout(); plt.show()
add_verdict("6 taste centroids", "mean user-user cosine", T6.loc["organic", "mean cosine"], T6.loc["recommended", "mean cosine"], "narrower")
add_verdict("6 taste centroids", "median nearest-neighbour cosine", res6["organic"], res6["recommended"], "narrower")

V = pd.DataFrame(verdict)
pd.set_option("display.width", 200)
print(V.to_string(index=False, float_format=lambda x: f"{x:,.4f}"))
print()
print(V.groupby("analysis")["recommended is"].value_counts().unstack(fill_value=0).to_string())

In [ ]:
def genre_counts(items, genre_of_row):
    U, _, n = items.shape
    G = genre_of_row[item_row[items]].astype(np.int64)
    K = int(genre_of_row.max()) + 1
    idx = (np.arange(U)[:, None, None] * 2 + np.arange(2)[None, :, None]) * K + G
    return np.bincount(idx.ravel(), minlength=U * 2 * K).reshape(U, 2, K)

def genre_metrics(items, genre_of_row):
    cnt = genre_counts(items, genre_of_row); n = items.shape[2]
    p = cnt / n
    ent = -np.where(p > 0, p * np.log2(np.where(p > 0, p, 1)), 0).sum(-1)
    novel_rec = (cnt[:, 0] * (cnt[:, 1] == 0)).sum(-1) / n      # recommended tracks in genres absent from the user's organic sample
    novel_org = (cnt[:, 1] * (cnt[:, 0] == 0)).sum(-1) / n
    return (cnt > 0).sum(-1), ent, novel_rec, novel_org, cnt

dg, ent, nov_rec, nov_org, gcnt = genre_metrics(items, genre_row)
res4 = {"distinct pseudo-genres": paired(dg[:, 1], dg[:, 0]), "genre entropy (bits)": paired(ent[:, 1], ent[:, 0]),
        "novel-genre share (recommended vs organic baseline)": paired(nov_org, nov_rec)}
print(pd.DataFrame(res4).T[["users", "organic", "recommended", "diff", "lo", "hi", "share_rec_higher", "p"]].to_string(float_format=lambda x: f"{x:.4g}"))

# robustness to the number of pseudo-genres
rows = []
for k in (16, 32, 64):
    gl = genre_row if k == K_GENRE else fit_genres(k)
    d_, e_, nr_, no_, _ = genre_metrics(items, gl)
    rows.append({"k": k, "distinct genres: organic": np.median(d_[:, 1]), "recommended": np.median(d_[:, 0]),
                 "users rec > org (distinct)": (d_[:, 0] > d_[:, 1]).mean(), "median entropy diff": np.median(e_[:, 0] - e_[:, 1]),
                 "novel share organic": np.median(no_), "novel share recommended": np.median(nr_)})
print("\nRobustness to k:"); print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.3f}"))

# which genres does each stream over-represent?
share = gcnt.sum(0) / (gcnt.shape[0] * N_TRACKS)                 # (2, K): rows 0 recommended, 1 organic
lr = np.log2(share[0] / share[1]); order = np.argsort(lr)
fig, ax = plt.subplots(1, 2, figsize=(13, 3.8))
ax[0].bar(range(K_GENRE), lr[order], color=np.where(lr[order] > 0, COL["recommended"], COL["organic"]))
ax[0].set_xticks(range(K_GENRE)); ax[0].set_xticklabels(order, fontsize=7)
ax[0].axhline(0, color="k", lw=1)
ax[0].set(title="Pseudo-genre share: log2(recommended / organic)", xlabel="pseudo-genre id", ylabel="log2 ratio")
ax[1].scatter(share[1] * 100, share[0] * 100, color="#264653")
mx = max(share.max() * 100, 1); ax[1].plot([0, mx], [0, mx], "k:", lw=1)
for g_ in np.argsort(np.abs(lr))[::-1][:5]:
    ax[1].annotate(str(g_), (share[1, g_] * 100, share[0, g_] * 100), fontsize=8)
ax[1].set(title="Share of tracks per pseudo-genre", xlabel="organic %", ylabel="recommended %")
plt.tight_layout(); plt.show()
add_verdict("4 pseudo-genres", "distinct pseudo-genres (median)", res4["distinct pseudo-genres"]["organic"], res4["distinct pseudo-genres"]["recommended"], "broader")
add_verdict("4 pseudo-genres", "genre entropy (median)", res4["genre entropy (bits)"]["organic"], res4["genre entropy (bits)"]["recommended"], "broader")
add_verdict("4 pseudo-genres", "novel-genre share", np.median(nov_org), np.median(nov_rec), "broader")

In [ ]:
# distinct real tracks per (user, stream), sorted by group = user*2 + stream
m = (ratio_arr >= REAL_PCT) & (erow_arr >= 0) & elig_u[uidx_arr]
key = ((uidx_arr[m].astype(np.int64) * 2 + org_arr[m]) << 32) | item_arr[m].astype(np.int64)
pool = np.unique(key)
pool_grp, pool_item = pool >> 32, pool & 0xFFFFFFFF
g_start = np.searchsorted(pool_grp, np.arange(2 * n_users))
g_size = np.searchsorted(pool_grp, np.arange(2 * n_users), side="right") - g_start

def shuffled_rows(groups, k):
    """For each basket group, k distinct random tracks of that user in that stream (embedding rows)."""
    st, sz = g_start[groups][:, None], g_size[groups][:, None]
    idx = st + rng.integers(0, sz, (len(groups), k))
    while True:                                  # redraw rows that contain a duplicate
        s = np.sort(idx, 1); bad = (np.diff(s, axis=1) == 0).any(1)
        if not bad.any(): break
        idx[bad] = st[bad] + rng.integers(0, sz[bad], (bad.sum(), k))
    return item_row[pool_item[idx]]

In [ ]:
def basket_metrics(rows_, chunk=20_000):
    k = rows_.shape[1]; iu = np.triu_indices(k, 1); out = []
    for s0 in range(0, len(rows_), chunk):
        r = rows_[s0:s0 + chunk]; Ev = EN[r]                       # (n, k, 128)
        G = np.einsum("nkd,njd->nkj", Ev, Ev)
        coh = G[:, iu[0], iu[1]].mean(1)                          # mean pairwise cosine
        G[:, np.arange(k), np.arange(k)] = -np.inf
        nn = G.max(2).mean(1)                                     # mean cosine to nearest other track in the session
        g = genre_row[r]; same = (g[:, iu[0]] == g[:, iu[1]]).mean(1)   # share of pairs in the same pseudo-genre
        out.append(np.c_[coh, nn, same])
    return np.vstack(out)

recs = []
for k, o, rc in chosen:
    for s, v, idxs in (("organic", 1, o), ("recommended", 0, rc)):
        real = item_row[bi_[start[idxs][:, None] + np.arange(k)]]
        shuf = shuffled_rows(b_user[idxs].astype(np.int64) * 2 + v, k)
        for kind, rows_ in (("real", real), ("shuffled", shuf)):
            M_ = basket_metrics(rows_)
            recs.append(pd.DataFrame({"stream": s, "kind": kind, "size": k, "user": b_user[idxs],
                                      "coherence": M_[:, 0], "nn_coherence": M_[:, 1], "same_genre": M_[:, 2]}))
BK = pd.concat(recs, ignore_index=True)

# per user: average, then excess = real - shuffled (how much the SESSION adds beyond the user's taste)
pu = BK.groupby(["user", "stream", "kind"])[["coherence", "nn_coherence", "same_genre"]].mean().unstack(["stream", "kind"])
n_b = BK[BK.kind == "real"].groupby(["user", "stream"]).size().unstack()
pu = pu.loc[n_b.index[(n_b >= 5).all(1)]]
res_b1 = {}
for met in ["coherence", "nn_coherence", "same_genre"]:
    for s in ["organic", "recommended"]:
        pu[(met + "_excess", s, "")] = pu[(met, s, "real")] - pu[(met, s, "shuffled")]
    res_b1[met + " (real)"] = paired(pu[(met, "organic", "real")].to_numpy(), pu[(met, "recommended", "real")].to_numpy())
    res_b1[met + " excess over shuffled"] = paired(pu[(met + "_excess", "organic", "")].to_numpy(),
                                                   pu[(met + "_excess", "recommended", "")].to_numpy())
print(pd.DataFrame(res_b1).T.to_string(float_format=lambda x: f"{x:.4g}"))

In [ ]:
REF = EN[rng.choice(len(EN), 50_000, replace=False)]
N_PAIRS = 50_000

def one_pair_per_basket(rows_):
    k = rows_.shape[1]; i = rng.integers(0, k, len(rows_)); j = (i + rng.integers(1, k, len(rows_))) % k
    n = np.arange(len(rows_)); return rows_[n, i], rows_[n, j]

def rank_pct(ra, rb, block=1000):
    """Share of reference tracks closer to a than b is (0 = b is a's nearest neighbour, 0.5 = random)."""
    cab = np.einsum("ij,ij->i", EN[ra], EN[rb]); out = np.empty(len(ra))
    for s0 in range(0, len(ra), block):
        out[s0:s0 + block] = ((EN[ra[s0:s0 + block]] @ REF.T) > cab[s0:s0 + block, None]).mean(1)
    return out

R2 = []
for s, v in (("organic", 1), ("recommended", 0)):
    A_, B_, Ash, Bsh = [], [], [], []
    for k, o, rc in chosen:
        idxs = o if v == 1 else rc
        a, b = one_pair_per_basket(item_row[bi_[start[idxs][:, None] + np.arange(k)]]); A_.append(a); B_.append(b)
        a, b = one_pair_per_basket(shuffled_rows(b_user[idxs].astype(np.int64) * 2 + v, k)); Ash.append(a); Bsh.append(b)
    A_, B_, Ash, Bsh = map(np.concatenate, (A_, B_, Ash, Bsh))
    pick = rng.choice(len(A_), min(N_PAIRS, len(A_)), replace=False)       # equal number of pairs per stream
    for kind, a, b in (("co-played", A_[pick], B_[pick]), ("shuffled", Ash[pick], Bsh[pick])):
        rp = rank_pct(a, b)
        R2.append({"stream": s, "pairs": kind, "median rank pct": np.median(rp),
                   "share in nearest 1%": (rp < 0.01).mean(), "share in nearest 5%": (rp < 0.05).mean()})
print(pd.DataFrame(R2).to_string(index=False, float_format=lambda x: f"{x:.4f}"))

In [ ]:
REF = EN[rng.choice(len(EN), 50_000, replace=False)]
N_PAIRS = 50_000

def one_pair_per_basket(rows_):
    k = rows_.shape[1]; i = rng.integers(0, k, len(rows_)); j = (i + rng.integers(1, k, len(rows_))) % k
    n = np.arange(len(rows_)); return rows_[n, i], rows_[n, j]

def rank_pct(ra, rb, block=1000):
    """Share of reference tracks closer to a than b is (0 = b is a's nearest neighbour, 0.5 = random)."""
    cab = np.einsum("ij,ij->i", EN[ra], EN[rb]); out = np.empty(len(ra))
    for s0 in range(0, len(ra), block):
        out[s0:s0 + block] = ((EN[ra[s0:s0 + block]] @ REF.T) > cab[s0:s0 + block, None]).mean(1)
    return out

R2 = []
for s, v in (("organic", 1), ("recommended", 0)):
    A_, B_, Ash, Bsh = [], [], [], []
    for k, o, rc in chosen:
        idxs = o if v == 1 else rc
        a, b = one_pair_per_basket(item_row[bi_[start[idxs][:, None] + np.arange(k)]]); A_.append(a); B_.append(b)
        a, b = one_pair_per_basket(shuffled_rows(b_user[idxs].astype(np.int64) * 2 + v, k)); Ash.append(a); Bsh.append(b)
    A_, B_, Ash, Bsh = map(np.concatenate, (A_, B_, Ash, Bsh))
    pick = rng.choice(len(A_), min(N_PAIRS, len(A_)), replace=False)       # equal number of pairs per stream
    for kind, a, b in (("co-played", A_[pick], B_[pick]), ("shuffled", Ash[pick], Bsh[pick])):
        rp = rank_pct(a, b)
        R2.append({"stream": s, "pairs": kind, "median rank pct": np.median(rp),
                   "share in nearest 1%": (rp < 0.01).mean(), "share in nearest 5%": (rp < 0.05).mean()})
print(pd.DataFrame(R2).to_string(index=False, float_format=lambda x: f"{x:.4f}"))